# Protein Inhibitor QSAR Demo
Estimate pIC50 from molecular structure and evaluate the model on chemical scaffolds held out from training. The example uses public ChEMBL acetylcholinesterase data. Run `python fetch_data.py` first.

## 1. Load the activity data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

path = Path("data/demo_activity.csv")
if not path.is_file():
    raise FileNotFoundError("Run python fetch_data.py from the repository root first.")
raw = pd.read_csv(path)
raw["pIC50"] = pd.to_numeric(raw["pIC50"], errors="coerce")
raw = raw.loc[np.isfinite(raw["pIC50"])].dropna(subset=["smiles"])
print(f"Loaded {len(raw)} usable activity rows.")

## 2. Clean structures and combine repeat measurements

In [ ]:
def canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)

raw = raw.assign(canonical=raw["smiles"].map(canonical_smiles))
data = raw.dropna(subset=["canonical"]).groupby("canonical", as_index=False)["pIC50"].median()
molecules = [Chem.MolFromSmiles(smiles) for smiles in data["canonical"]]
print(f"{len(data)} unique, valid single-fragment structures.")

## 3. Generate Morgan fingerprints and scaffold groups

In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def scaffold_group(mol):
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return scaffold or "<acyclic>"

groups = np.array([scaffold_group(mol) for mol in molecules])
if len(np.unique(groups)) < 5:
    raise ValueError("At least five scaffold groups are needed for this demo.")
X = np.stack([generator.GetFingerprintAsNumPy(mol) for mol in molecules])
y = data["pIC50"].to_numpy()
print(f"{X.shape[0]} molecules, {X.shape[1]} fingerprint bits, {len(np.unique(groups))} groups.")

## 4. Hold out entire scaffolds and fit two models
The test split contains about 20% of the scaffold groups. All molecules with the same scaffold stay together. The baseline predicts the average training pIC50.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train, test = next(splitter.split(X, y, groups))
assert not set(groups[train]).intersection(groups[test])

models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
}
predictions = {}
for name, model in models.items():
    model.fit(X[train], y[train])
    predictions[name] = model.predict(X[test])

print(f"Train: {len(train)} molecules. Test: {len(test)} molecules.")

## 5. Report held-out error and plot predictions

In [ ]:
scores = pd.DataFrame([
    {
        "Model": name,
        "MAE": mean_absolute_error(y[test], predicted),
        "RMSE": np.sqrt(mean_squared_error(y[test], predicted)),
    }
    for name, predicted in predictions.items()
]).set_index("Model")
display(scores.round(3))

observed = y[test]
predicted = predictions["Random forest"]
lo = min(observed.min(), predicted.min())
hi = max(observed.max(), predicted.max())
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(observed, predicted, alpha=0.65)
ax.plot([lo, hi], [lo, hi], "--", color="0.5")
ax.set(xlabel="Observed pIC50", ylabel="Predicted pIC50", title="Held-out scaffolds")
fig.tight_layout()
plt.show()

## 6. Interpret the results
Compare the random forest's MAE and RMSE with the baseline. Lower values indicate smaller prediction errors. The plot shows where the model overestimates or underestimates potency. Results depend on the activity records and the scaffold split. Testing additional splits or an independent dataset would give a broader assessment.